# Brain Tumor MRI Classification Project

## CNN From Scratch vs ResNet50 Transfer Learning

This notebook demonstrates the full workflow for classifying brain MRI images into four classes:

- Glioma
- Meningioma
- No Tumor
- Pituitary Tumor

The notebook covers dataset checking, preprocessing, CNN model, ResNet50 transfer learning, training, evaluation, model comparison, Grad-CAM, and Streamlit app usage.

> Place the dataset in `data/raw/brain_dataset/` and trained models in `models/` before running all cells.

## 1. Project Folder Structure

```text
brain-tumor-classification/
├── app/app.py
├── src/data/dataset_loader.py
├── src/data/preprocessing.py
├── src/models/cnn_from_scratch.py
├── src/models/transfer_learning.py
├── src/train.py
├── src/evaluate.py
├── data/raw/brain_dataset/Training/
├── data/raw/brain_dataset/Testing/
├── models/
├── results/
├── logs/
├── compare_models.py
├── gradcam.py
├── requirements.txt
└── README.md
```

## 2. Install Requirements

Run this only if dependencies are not installed. TensorFlow works best with Python 3.11 or 3.12.

In [ ]:
# !pip install -r requirements.txt

## 3. Import Libraries

In [ ]:
import os
import json
import time
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
    BatchNormalization, GlobalAveragePooling2D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, TensorBoard

from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score
)

print("TensorFlow version:", tf.__version__)

## 4. Define Project Paths

In [ ]:
BASE_DIR = os.getcwd()

DATA_DIR = os.path.join(BASE_DIR, "data", "raw", "brain_dataset")
TRAIN_DIR = os.path.join(DATA_DIR, "Training")
TEST_DIR = os.path.join(DATA_DIR, "Testing")

MODEL_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
LOGS_DIR = os.path.join(BASE_DIR, "logs")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Training directory:", TRAIN_DIR)
print("Testing directory:", TEST_DIR)

## 5. Check Dataset Structure

This checks if the expected Training and Testing folders exist and counts images in every class.

In [ ]:
def check_dataset_structure(train_dir=TRAIN_DIR, test_dir=TEST_DIR):
    def check_folder(folder, name):
        if not os.path.exists(folder):
            print(f"{name} folder NOT found:", folder)
            return False

        classes = sorted([item for item in os.listdir(folder) if os.path.isdir(os.path.join(folder, item))])
        print(f"\n{name} folder found with {len(classes)} classes:")

        total_images = 0
        for cls in classes:
            class_path = os.path.join(folder, cls)
            count = len([file for file in os.listdir(class_path) if file.lower().endswith((".jpg", ".jpeg", ".png"))])
            total_images += count
            print(f"  - {cls}: {count} images")

        print(f"Total images in {name}: {total_images}")
        return True

    train_ok = check_folder(train_dir, "Training")
    test_ok = check_folder(test_dir, "Testing")

    if train_ok and test_ok:
        print("\nDataset folders are ready.")
    else:
        print("\nPlease place the dataset in data/raw/brain_dataset/")

check_dataset_structure()

## 6. Check Corrupted Images

In [ ]:
def check_corrupted_images(folder):
    corrupted_images = []

    if not os.path.exists(folder):
        print("Folder not found:", folder)
        return corrupted_images

    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                image_path = os.path.join(root, file)
                try:
                    with Image.open(image_path) as img:
                        img.verify()
                except Exception:
                    corrupted_images.append(image_path)

    print(f"Corrupted images found in {folder}: {len(corrupted_images)}")
    for img in corrupted_images[:20]:
        print("Corrupted:", img)

    return corrupted_images

train_corrupted = check_corrupted_images(TRAIN_DIR)
test_corrupted = check_corrupted_images(TEST_DIR)

## 7. Preprocessing and Data Generators

CNN uses pixel scaling to `[0, 1]`. ResNet50 uses `preprocess_input`. Both use augmentation during training.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def get_generators(use_resnet=True):
    if not os.path.exists(TRAIN_DIR):
        raise FileNotFoundError(f"Training directory not found: {TRAIN_DIR}")
    if not os.path.exists(TEST_DIR):
        raise FileNotFoundError(f"Testing directory not found: {TEST_DIR}")

    if use_resnet:
        train_datagen = ImageDataGenerator(
            preprocessing_function=preprocess_input,
            rotation_range=20,
            width_shift_range=0.2,
            height_shift_range=0.2,
            zoom_range=0.2,
            horizontal_flip=True,
            brightness_range=[0.8, 1.2],
            validation_split=0.15
        )
        test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
    else:
        train_datagen = ImageDataGenerator(
            rescale=1.0 / 255,
            rotation_range=20,
            width_shift_range=0.2,
            height_shift_range=0.2,
            zoom_range=0.2,
            horizontal_flip=True,
            brightness_range=[0.8, 1.2],
            validation_split=0.15
        )
        test_datagen = ImageDataGenerator(rescale=1.0 / 255)

    train_gen = train_datagen.flow_from_directory(
        TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode="categorical", shuffle=True, subset="training"
    )

    val_gen = train_datagen.flow_from_directory(
        TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode="categorical", shuffle=False, subset="validation"
    )

    test_gen = test_datagen.flow_from_directory(
        TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode="categorical", shuffle=False
    )

    return train_gen, val_gen, test_gen

## 8. Preview Sample Images

In [ ]:
def show_sample_images(folder=TRAIN_DIR, images_per_class=3):
    if not os.path.exists(folder):
        print("Folder not found:", folder)
        return

    classes = sorted([item for item in os.listdir(folder) if os.path.isdir(os.path.join(folder, item))])

    for cls in classes:
        class_path = os.path.join(folder, cls)
        image_files = [file for file in os.listdir(class_path) if file.lower().endswith((".jpg", ".jpeg", ".png"))][:images_per_class]

        if not image_files:
            continue

        plt.figure(figsize=(12, 3))
        plt.suptitle(cls.upper(), fontsize=14)

        for i, file in enumerate(image_files):
            img = Image.open(os.path.join(class_path, file)).convert("RGB")
            plt.subplot(1, images_per_class, i + 1)
            plt.imshow(img)
            plt.axis("off")

        plt.show()

show_sample_images()

## 9. CNN From Scratch Model

In [ ]:
def build_cnn_from_scratch(input_shape=(224, 224, 3), num_classes=4):
    model = Sequential([
        Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        Conv2D(64, (3, 3), activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        Conv2D(128, (3, 3), activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        Flatten(),
        Dense(256, activation="relu"),
        Dropout(0.5),
        Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

cnn_model = build_cnn_from_scratch(num_classes=4)
cnn_model.summary()

## 10. ResNet50 Transfer Learning Model

In [ ]:
def build_transfer_model(input_shape=(224, 224, 3), num_classes=4, fine_tune=True):
    base_model = ResNet50(weights="imagenet", include_top=False, input_shape=input_shape)

    for layer in base_model.layers:
        layer.trainable = False

    if fine_tune:
        for layer in base_model.layers[-30:]:
            layer.trainable = True

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=base_model.input, outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# Uncomment to build and view the ResNet50 model.
# resnet_model = build_transfer_model(num_classes=4)
# resnet_model.summary()

## 11. Training Function

Training is commented in the next cells to avoid running heavy work automatically.

In [ ]:
def plot_history(history, model_name):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history.history["accuracy"], label="Train Accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
    plt.title(f"{model_name} Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.title(f"{model_name} Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()
    path = os.path.join(RESULTS_DIR, f"{model_name}_history.png")
    plt.savefig(path)
    plt.show()

def count_trainable_params(model):
    return int(sum([w.shape.num_elements() for w in model.trainable_weights]))

def train_model(model_name, model, train_gen, val_gen, epochs=5):
    best_model_path = os.path.join(MODEL_DIR, f"best_{model_name}.h5")
    final_model_path = os.path.join(MODEL_DIR, f"final_{model_name}.h5")

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
        ModelCheckpoint(filepath=best_model_path, monitor="val_accuracy", save_best_only=True, verbose=1),
        TensorBoard(log_dir=os.path.join(LOGS_DIR, f"{model_name}_{datetime.now().strftime('%Y%m%d-%H%M%S')}"))
    ]

    start_time = time.time()
    history = model.fit(train_gen, validation_data=val_gen, epochs=epochs, callbacks=callbacks, verbose=1)
    training_time = time.time() - start_time

    model.save(final_model_path)
    plot_history(history, model_name)

    metrics = {
        "model_name": model_name,
        "best_train_accuracy": float(max(history.history["accuracy"])),
        "best_val_accuracy": float(max(history.history["val_accuracy"])),
        "final_train_accuracy": float(history.history["accuracy"][-1]),
        "final_val_accuracy": float(history.history["val_accuracy"][-1]),
        "training_time_seconds": round(training_time, 2),
        "trainable_parameters": count_trainable_params(model)
    }

    with open(os.path.join(RESULTS_DIR, f"{model_name}_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=4)

    with open(os.path.join(RESULTS_DIR, f"{model_name}_history.json"), "w") as f:
        json.dump(history.history, f, indent=4)

    print(f"{model_name} training completed.")
    return metrics

## 12. Train Models

Uncomment the model you want to train.

In [ ]:
# Train CNN
# train_gen_cnn, val_gen_cnn, test_gen_cnn = get_generators(use_resnet=False)
# cnn_model = build_cnn_from_scratch(num_classes=train_gen_cnn.num_classes)
# cnn_metrics = train_model("CNN_From_Scratch", cnn_model, train_gen_cnn, val_gen_cnn, epochs=5)

# Train ResNet50
# train_gen_resnet, val_gen_resnet, test_gen_resnet = get_generators(use_resnet=True)
# resnet_model = build_transfer_model(num_classes=train_gen_resnet.num_classes)
# resnet_metrics = train_model("ResNet50_Transfer", resnet_model, train_gen_resnet, val_gen_resnet, epochs=5)

## 13. Evaluation Function

In [ ]:
def evaluate_model(model_name, model_path, test_gen, class_names):
    if not os.path.exists(model_path):
        print(f"Model not found: {model_path}")
        return None

    model = load_model(model_path)
    test_gen.reset()

    predictions = model.predict(test_gen, verbose=1)
    y_pred = np.argmax(predictions, axis=1)
    y_true = test_gen.classes

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    report = classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0)
    print(report)

    with open(os.path.join(RESULTS_DIR, f"{model_name}_classification_report.txt"), "w") as f:
        f.write(report)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    plt.imshow(cm)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(range(len(class_names)), class_names, rotation=45)
    plt.yticks(range(len(class_names)), class_names)

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.tight_layout()
    cm_path = os.path.join(RESULTS_DIR, f"{model_name}_confusion_matrix.png")
    plt.savefig(cm_path)
    plt.show()

    results = {
        "Model": model_name,
        "Accuracy": round(float(acc), 4),
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1 Score": round(float(f1), 4)
    }

    with open(os.path.join(RESULTS_DIR, f"{model_name}_evaluation_metrics.json"), "w") as f:
        json.dump(results, f, indent=4)

    return results

## 14. Evaluate Saved Models

Run after training or after placing `.h5` files in `models/`.

In [ ]:
# CNN evaluation
# _, _, test_gen_cnn = get_generators(use_resnet=False)
# class_names_cnn = list(test_gen_cnn.class_indices.keys())
# cnn_result = evaluate_model("CNN_From_Scratch", os.path.join(MODEL_DIR, "best_CNN_From_Scratch.h5"), test_gen_cnn, class_names_cnn)

# ResNet50 evaluation
# _, _, test_gen_resnet = get_generators(use_resnet=True)
# class_names_resnet = list(test_gen_resnet.class_indices.keys())
# resnet_result = evaluate_model("ResNet50_Transfer", os.path.join(MODEL_DIR, "best_ResNet50_Transfer.h5"), test_gen_resnet, class_names_resnet)

## 15. Compare Models

In [ ]:
def load_json(path):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {}

def compare_models():
    cnn_eval = load_json(os.path.join(RESULTS_DIR, "CNN_From_Scratch_evaluation_metrics.json"))
    resnet_eval = load_json(os.path.join(RESULTS_DIR, "ResNet50_Transfer_evaluation_metrics.json"))
    cnn_train = load_json(os.path.join(RESULTS_DIR, "CNN_From_Scratch_metrics.json"))
    resnet_train = load_json(os.path.join(RESULTS_DIR, "ResNet50_Transfer_metrics.json"))

    data = [
        {
            "Model": "CNN From Scratch",
            "Accuracy": cnn_eval.get("Accuracy", "N/A"),
            "Precision": cnn_eval.get("Precision", "N/A"),
            "Recall": cnn_eval.get("Recall", "N/A"),
            "F1 Score": cnn_eval.get("F1 Score", "N/A"),
            "Best Train Accuracy": cnn_train.get("best_train_accuracy", "N/A"),
            "Best Validation Accuracy": cnn_train.get("best_val_accuracy", "N/A"),
            "Training Time Seconds": cnn_train.get("training_time_seconds", "N/A"),
            "Trainable Parameters": cnn_train.get("trainable_parameters", "N/A")
        },
        {
            "Model": "ResNet50 Transfer",
            "Accuracy": resnet_eval.get("Accuracy", "N/A"),
            "Precision": resnet_eval.get("Precision", "N/A"),
            "Recall": resnet_eval.get("Recall", "N/A"),
            "F1 Score": resnet_eval.get("F1 Score", "N/A"),
            "Best Train Accuracy": resnet_train.get("best_train_accuracy", "N/A"),
            "Best Validation Accuracy": resnet_train.get("best_val_accuracy", "N/A"),
            "Training Time Seconds": resnet_train.get("training_time_seconds", "N/A"),
            "Trainable Parameters": resnet_train.get("trainable_parameters", "N/A")
        }
    ]

    df = pd.DataFrame(data)
    df.to_csv(os.path.join(RESULTS_DIR, "model_comparison.csv"), index=False)
    display(df)

    metrics = ["Accuracy", "Precision", "Recall", "F1 Score"]
    plot_df = df[["Model"] + metrics].copy()

    for metric in metrics:
        plot_df[metric] = pd.to_numeric(plot_df[metric], errors="coerce")

    plot_df.set_index("Model")[metrics].plot(kind="bar", figsize=(10, 6))
    plt.title("Model Evaluation Comparison")
    plt.ylabel("Score")
    plt.ylim(0, 1)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "model_comparison_chart.png"))
    plt.show()

    return df

comparison_df = compare_models()

## 16. Single Image Prediction

In [ ]:
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]

def preprocess_for_cnn(image):
    image = image.convert("RGB").resize(IMG_SIZE)
    img_array = np.array(image, dtype=np.float32) / 255.0
    return np.expand_dims(img_array, axis=0)

def preprocess_for_resnet(image):
    image = image.convert("RGB").resize(IMG_SIZE)
    img_array = np.array(image, dtype=np.float32)
    img_array = np.expand_dims(img_array, axis=0)
    return preprocess_input(img_array)

def predict_model(model, image_array):
    prediction = model.predict(image_array, verbose=0)
    predicted_index = int(np.argmax(prediction))
    predicted_class = CLASS_NAMES[predicted_index]
    confidence = float(np.max(prediction)) * 100
    probabilities = {CLASS_NAMES[i]: float(prediction[0][i]) * 100 for i in range(len(CLASS_NAMES))}
    return predicted_class, confidence, probabilities

def predict_single_image(img_path):
    if not os.path.exists(img_path):
        print("Image not found:", img_path)
        return

    image = Image.open(img_path).convert("RGB")
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.axis("off")
    plt.title("Input MRI Image")
    plt.show()

    results = []
    cnn_path = os.path.join(MODEL_DIR, "best_CNN_From_Scratch.h5")
    resnet_path = os.path.join(MODEL_DIR, "best_ResNet50_Transfer.h5")

    if os.path.exists(cnn_path):
        cnn_model = load_model(cnn_path)
        cls, conf, probs = predict_model(cnn_model, preprocess_for_cnn(image))
        results.append({"Model": "CNN From Scratch", "Prediction": cls, "Confidence": f"{conf:.2f}%"})

    if os.path.exists(resnet_path):
        resnet_model = load_model(resnet_path)
        cls, conf, probs = predict_model(resnet_model, preprocess_for_resnet(image))
        results.append({"Model": "ResNet50 Transfer", "Prediction": cls, "Confidence": f"{conf:.2f}%"})

    if results:
        display(pd.DataFrame(results))
    else:
        print("No saved models found in models/ folder.")

# Example:
# predict_single_image("data/raw/brain_dataset/Testing/glioma/example.jpg")

## 17. Grad-CAM Explainability

Grad-CAM highlights image regions that influenced the ResNet50 prediction.

In [ ]:
import cv2

def get_img_array(img_path, size=(224, 224)):
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=size)
    array = tf.keras.preprocessing.image.img_to_array(img)
    array = np.expand_dims(array, axis=0)
    return preprocess_input(array)

def make_gradcam_heatmap(img_array, model, last_conv_layer_name=None):
    if last_conv_layer_name is None:
        for layer in reversed(model.layers):
            if "conv" in layer.name:
                last_conv_layer_name = layer.name
                break

    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index = tf.argmax(predictions[0])
        loss = predictions[:, pred_index]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = np.maximum(heatmap, 0)
    heatmap = heatmap / (np.max(heatmap) + 1e-8)
    return heatmap.numpy(), last_conv_layer_name

def show_gradcam(img_path):
    model_path = os.path.join(MODEL_DIR, "best_ResNet50_Transfer.h5")
    if not os.path.exists(model_path):
        print("ResNet50 model not found:", model_path)
        return
    if not os.path.exists(img_path):
        print("Image not found:", img_path)
        return

    model = load_model(model_path)
    img_array = get_img_array(img_path)
    heatmap, layer_name = make_gradcam_heatmap(img_array, model)
    print("Using layer:", layer_name)

    img = cv2.imread(img_path)
    img = cv2.resize(img, IMG_SIZE)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    heatmap = cv2.resize(heatmap, IMG_SIZE)
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    superimposed = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(superimposed)
    plt.title("Grad-CAM")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# Example:
# show_gradcam("data/raw/brain_dataset/Testing/glioma/example.jpg")

## 18. Streamlit App

To run the app from terminal:

```bash
streamlit run app/app.py
```

The app uploads one MRI image and compares predictions from CNN and ResNet50.

## 19. Discussion Summary

### Problem Statement

This project classifies brain MRI images into glioma, meningioma, pituitary tumor, and no tumor.

### Models

- **CNN From Scratch:** baseline model built manually.
- **ResNet50 Transfer Learning:** pretrained model with a custom classification head.

### Evaluation

Models are compared using accuracy, precision, recall, F1-score, classification report, and confusion matrix.

### Conclusion

ResNet50 transfer learning is expected to perform better because it uses pretrained visual features, while CNN from scratch learns everything only from the project dataset.